# Phase 2: Manifold Learning & Visualization - Colab Edition

## Instructions for Using This Notebook in Google Colab:

### Step 1: Upload Your Data to Google Drive
1. Open Google Drive in your browser
2. Create a folder named `chromatin_data`
3. Upload your data files to this folder:
   - `train_sequences.csv` - DNA sequences (one per line, no header)
   - `train_labels.csv` - Labels (one per line, no header)

### Step 2: Mount Google Drive
Run the mounting cell below when prompted

### Step 3: Configure Settings
- Set `USE_DEMO_DATA = False` in the configuration cell to use your data
- Or set `USE_DEMO_DATA = True` to test with synthetic data

### Step 4: Run All Cells
Click "Runtime" -> "Run all" or run cells sequentially

---

## Pipeline Overview:
1. **Feature Extraction** - K-mer frequencies, positional profiles, dinucleotide features
2. **Dimensionality Reduction** - PCA, UMAP, PHATE embeddings
3. **Cluster Analysis** - Silhouette scores, ARI, hierarchical clustering  
4. **Visualization** - 2D scatter plots, comparison charts, variance analysis

## Section 1: Environment Setup

In [ ]:
# @title Install Required Packages
!pip install -q numpy pandas scikit-learn umap-learn phate matplotlib seaborn tqdm joblib scipy torch

print("\n=== Packages Installed Successfully ===")
print("- numpy, pandas: Data handling")
print("- scikit-learn: PCA, clustering, metrics")
print("- umap-learn: UMAP dimensionality reduction")
print("- phate: PHATE dimensionality reduction")
print("- matplotlib, seaborn: Visualization")
print("- tqdm: Progress bars")
print("- joblib: Parallel processing")
print("- torch: GPU detection")

In [ ]:
# @title Import Libraries
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from functools import partial
from typing import Optional, Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score, silhouette_samples
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform

# Dimensionality reduction
import umap
import phate

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from joblib import Parallel, delayed, cpu_count

# Check hardware
import torch
HAS_CUDA = torch.cuda.is_available()
N_CPUS = cpu_count()

print("\n=== Environment Information ===")
print(f"CPU cores available: {N_CPUS}")
print(f"GPU available: {HAS_CUDA}")
if HAS_CUDA:
    print(f"  GPU name: {torch.cuda.get_device_name(0)}")
print("\nAll libraries imported successfully!")

## Section 2: Google Drive Mounting and Configuration

In [ ]:
# @title Mount Google Drive
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')
print("\nGoogle Drive mounted at: /content/drive")

In [ ]:
# @title Configuration
# ============

# @markdown **Data Source Configuration:**
USE_DEMO_DATA = True  # @param {type:"boolean"}

# @markdown **Paths:**
DATA_PATH = '/content/drive/MyDrive/chromatin_data'  # @param {type:"string"}
OUTPUT_PATH = '/content/drive/MyDrive/phase2_results'  # @param {type:"string"}

# @markdown **Data Subsampling (for faster testing):**
SUBSAMPLE_SIZE = None  # @param {type:"integer"}

# @markdown **Feature Extraction Settings:**
KMER_K = 5  # @param {type:"integer"}
N_POSITIONAL_BINS = 10  # @param {type:"integer"}
FEATURE_EXTRACTION_BATCH_SIZE = 1000  # @param {type:"integer"}

# @markdown **Dimensionality Reduction Settings:**
PCA_N_COMPONENTS = 50  # @param {type:"integer"}
USE_PCA_PREPROCESSING = True  # @param {type:"boolean"}
N_PCS_FOR_PHATE = 30  # @param {type:"integer"}

# @markdown **Parallel Processing:**
N_JOBS = -1  # @param {type:"integer"}

# @markdown **Random Seed:**
RANDOM_STATE = 42  # @param {type:"integer"}

# UMAP and PHATE parameter presets
UMAP_PARAMS = [
    {"n_neighbors": 15, "min_dist": 0.1},
    {"n_neighbors": 30, "min_dist": 0.1},
    {"n_neighbors": 15, "min_dist": 0.25},
    {"n_neighbors": 50, "min_dist": 0.0}
]

PHATE_PARAMS = [
    {"knn": 10, "decay": 20},
    {"knn": 15, "decay": 40},
    {"knn": 5, "decay": 10}
]

print("\n=== Configuration Summary ===")
print(f"Use demo data: {USE_DEMO_DATA}")
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Subsample size: {SUBSAMPLE_SIZE if SUBSAMPLE_SIZE else 'All data'}")
print(f"K-mer size: {KMER_K}")
print(f"Parallel jobs: {N_JOBS if N_JOBS != -1 else 'All available (' + str(N_CPUS) + ')'}")

## Section 3: Data Loading

In [ ]:
# @title Load Data
def load_data(data_path: str, use_demo: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if use_demo:
        print("Generating demo data for testing...")
        np.random.seed(RANDOM_STATE)
        n_samples = 10000
        seq_length = 200
        bases = ['A', 'C', 'G', 'T']
        sequences = [''.join(np.random.choice(bases, seq_length)) for _ in range(n_samples)]
        labels = np.random.randint(1, 19, n_samples)
        return pd.DataFrame(sequences, columns=['sequence']), pd.DataFrame(labels, columns=['label'])
    else:
        seq_file = os.path.join(data_path, 'train_sequences.csv')
        label_file = os.path.join(data_path, 'train_labels.csv')
        if not os.path.exists(seq_file):
            raise FileNotFoundError(f"Sequence file not found: {seq_file}")
        if not os.path.exists(label_file):
            raise FileNotFoundError(f"Label file not found: {label_file}")
        print(f"Loading sequences from {seq_file}...")
        sequences_df = pd.read_csv(seq_file, header=None, names=['sequence'])
        print(f"Loading labels from {label_file}...")
        labels_df = pd.read_csv(label_file, header=None, names=['label'])
        return sequences_df, labels_df

sequences_df, labels_df = load_data(DATA_PATH, USE_DEMO_DATA)
print(f"\nLoaded {len(sequences_df)} sequences")
print(f"Unique labels: {sorted(labels_df['label'].unique())}")
print(f"\nLabel distribution:\n{labels_df['label'].value_counts().sort_index()}")

In [ ]:
# @title Subsample Data (Optional)
if SUBSAMPLE_SIZE and SUBSAMPLE_SIZE < len(sequences_df):
    print(f"Subsampling {SUBSAMPLE_SIZE} sequences...")
    np.random.seed(RANDOM_STATE)
    indices = np.random.choice(len(sequences_df), SUBSAMPLE_SIZE, replace=False)
    sequences_df = sequences_df.iloc[indices].reset_index(drop=True)
    labels_df = labels_df.iloc[indices].reset_index(drop=True)
    print(f"Subsampled to {len(sequences_df)} sequences")

sequences = sequences_df['sequence'].tolist()
labels = labels_df['label'].values
print(f"\nReady: {len(sequences)} sequences, {len(labels)} labels")

## Section 4: Feature Extraction

In [ ]:
# @title Define Feature Extraction Functions
def generate_kmer_vocabulary(k: int) -> Dict[str, int]:
    bases = ['A', 'C', 'G', 'T']
    kmers = [''.join(p) for p in product(bases, repeat=k)]
    return {kmer: idx for idx, kmer in enumerate(kmers)}

def compute_kmer_frequencies(sequence: str, k: int, vocab: Dict[str, int]) -> np.ndarray:
    counts = np.zeros(len(vocab), dtype=np.float32)
    n_kmers = len(sequence) - k + 1
    if n_kmers <= 0:
        return counts
    for i in range(n_kmers):
        kmer = sequence[i:i+k]
        if kmer in vocab:
            counts[vocab[kmer]] += 1
    total = counts.sum()
    if total > 0:
        counts /= total
    return counts

def compute_positional_kmer_profiles(sequence: str, k: int, vocab: Dict[str, int], n_bins: int = 10) -> np.ndarray:
    seq_len = len(sequence)
    bin_size = seq_len // n_bins
    profiles = np.zeros((n_bins, len(vocab)), dtype=np.float32)
    for bin_idx in range(n_bins):
        start = bin_idx * bin_size
        end = start + bin_size if bin_idx < n_bins - 1 else seq_len
        profiles[bin_idx] = compute_kmer_frequencies(sequence[start:end], k, vocab)
    return profiles.flatten()

def compute_dinucleotide_frequencies(sequence: str) -> np.ndarray:
    dinuc_vocab = generate_kmer_vocabulary(2)
    return compute_kmer_frequencies(sequence, 2, dinuc_vocab)

print("Feature extraction functions defined!")

In [ ]:
# @title Extract All Features
def process_single_sequence(seq: str, k: int, vocab: Dict[str, int], n_bins: int):
    if not seq or len(seq) < k:
        vocab_size = len(vocab)
        return (np.zeros(vocab_size, dtype=np.float32),
                np.zeros(n_bins * vocab_size, dtype=np.float32),
                np.zeros(16, dtype=np.float32), False)
    return (compute_kmer_frequencies(seq, k, vocab),
            compute_positional_kmer_profiles(seq, k, vocab, n_bins),
            compute_dinucleotide_frequencies(seq), True)

print("Generating k-mer vocabulary...")
vocab = generate_kmer_vocabulary(KMER_K)
print(f"Vocabulary size: {len(vocab)} ({KMER_K}-mers)")

from time import time
start_time = time()

n_samples = len(sequences)
vocab_size = len(vocab)
kmer_features = np.zeros((n_samples, vocab_size), dtype=np.float32)
positional_features = np.zeros((n_samples, N_POSITIONAL_BINS * vocab_size), dtype=np.float32)
dinuc_features = np.zeros((n_samples, 16), dtype=np.float32)

print(f"Processing {n_samples} sequences with {N_JOBS if N_JOBS != -1 else N_CPUS} parallel jobs...")
process_func = partial(process_single_sequence, k=KMER_K, vocab=vocab, n_bins=N_POSITIONAL_BINS)
results = Parallel(n_jobs=N_JOBS, batch_size=FEATURE_EXTRACTION_BATCH_SIZE, verbose=10)(
    delayed(process_func)(seq) for seq in tqdm(sequences, desc="Extracting features")
)

skipped = 0
for i, (kmer_feat, pos_feat, dinuc_feat, is_valid) in enumerate(results):
    if not is_valid:
        skipped += 1
    else:
        kmer_features[i] = kmer_feat
        positional_features[i] = pos_feat
        dinuc_features[i] = dinuc_feat

elapsed = time() - start_time
print(f"\nFeature extraction complete in {elapsed:.1f}s ({skipped} skipped)")
print(f"K-mer features: {kmer_features.shape}")
print(f"Positional features: {positional_features.shape}")
print(f"Dinucleotide features: {dinuc_features.shape}")

In [ ]:
# @title Save Features
output_dir = Path(OUTPUT_PATH)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving features to {output_dir}...")
np.save(output_dir / f"kmer_{KMER_K}_features.npy", kmer_features)
np.save(output_dir / "positional_kmer_features.npy", positional_features)
np.save(output_dir / "dinucleotide_features.npy", dinuc_features)
np.save(output_dir / "labels.npy", labels)
with open(output_dir / f"kmer_{KMER_K}_vocab.json", 'w') as f:
    json.dump(vocab, f)
print("Features saved successfully!")

## Section 5: Dimensionality Reduction

In [ ]:
# @title Define Dimensionality Reduction Functions
def run_pca(features: np.ndarray, n_components: int = 50) -> Dict:
    print(f"\nRunning PCA with {n_components} components...")
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    pca = PCA(n_components=n_components)
    embeddings = pca.fit_transform(features_scaled)
    result = {'embeddings': embeddings, 'embeddings_2d': embeddings[:, :2],
               'explained_variance_ratio': pca.explained_variance_ratio_,
               'cumulative_variance': np.cumsum(pca.explained_variance_ratio_)}
    print(f"Variance explained by 2 PCs: {result['cumulative_variance'][1]:.3f}")
    return result

def run_umap(features: np.ndarray, n_neighbors: int = 15, min_dist: float = 0.1) -> Dict:
    print(f"\nRunning UMAP (n_neighbors={n_neighbors}, min_dist={min_dist})...")
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, n_components=2,
                          metric='euclidean', n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=True)
    embeddings = reducer.fit_transform(features)
    return {'embeddings': embeddings, 'embeddings_2d': embeddings}

def run_phate_with_pca(features: np.ndarray, knn: int = 10, decay: int = 20) -> Dict:
    print(f"\nRunning PHATE with PCA pre-processing (knn={knn}, decay={decay})...")
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    pca = PCA(n_components=N_PCS_FOR_PHATE, random_state=RANDOM_STATE)
    features_reduced = pca.fit_transform(features_scaled)
    print(f"PCA reduced: {features.shape} -> {features_reduced.shape}")
    phate_op = phate.PHATE(knn=knn, decay=decay, n_components=2, n_jobs=N_JOBS, verbose=1)
    embeddings = phate_op.fit_transform(features_reduced)
    return {'embeddings': embeddings, 'embeddings_2d': embeddings}

print("Dimensionality reduction functions defined!")

In [ ]:
# @title Run All Dimensionality Reduction Methods
features = kmer_features
print(f"Using k-mer features: {features.shape}")

print("\n" + "="*60)
print("STEP 2: DIMENSIONALITY REDUCTION")
print("="*60)

all_embeddings = {}
start_total = time()

# PCA
print("\n--- PCA ---")
pca_result = run_pca(features, n_components=PCA_N_COMPONENTS)
all_embeddings['pca'] = pca_result

# UMAP
print("\n--- UMAP ---")
for i, params in enumerate(UMAP_PARAMS):
    print(f"\nUMAP config {i+1}/{len(UMAP_PARAMS)}")
    umap_result = run_umap(features, n_neighbors=params['n_neighbors'], min_dist=params['min_dist'])
    key = f"umap_n{params['n_neighbors']}_d{params['min_dist']}"
    all_embeddings[key] = umap_result

# PHATE
print("\n--- PHATE ---")
for i, params in enumerate(PHATE_PARAMS):
    print(f"\nPHATE config {i+1}/{len(PHATE_PARAMS)}")
    phate_result = run_phate_with_pca(features, knn=params['knn'], decay=params['decay'])
    key = f"phate_k{params['knn']}_d{params['decay']}_pca{N_PCS_FOR_PHATE}"
    all_embeddings[key] = phate_result

total_time = time() - start_total
print(f"\n{'='*60}")
print(f"Dimensionality reduction complete in {total_time:.1f}s")
print(f"Generated {len(all_embeddings)} embeddings: {list(all_embeddings.keys())}")

In [ ]:
# @title Save Embeddings
print(f"\nSaving embeddings to {output_dir}...")
for name, result in all_embeddings.items():
    if 'embeddings_2d' in result:
        np.save(output_dir / f"{name}.npy", result['embeddings_2d'])
np.save(output_dir / "pca_embeddings.npy", pca_result['embeddings'])
np.save(output_dir / "pca_variance_ratio.npy", pca_result['explained_variance_ratio'])
print("Embeddings saved successfully!")

## Section 6: Cluster Analysis

In [ ]:
# @title Define Cluster Analysis Functions
def compute_silhouette_scores(embeddings: np.ndarray, labels: np.ndarray) -> Dict:
    overall = silhouette_score(embeddings, labels)
    sample_scores = silhouette_samples(embeddings, labels)
    unique_labels = np.unique(labels)
    per_class = {int(l): float(np.mean(sample_scores[labels == l])) for l in unique_labels}
    sorted_classes = sorted(per_class.items(), key=lambda x: x[1], reverse=True)
    print(f"Overall silhouette: {overall:.4f}")
    return {'overall': float(overall), 'per_class': per_class,
            'well_clustered': [c[0] for c in sorted_classes[:5]],
            'dispersed': [c[0] for c in sorted_classes[-5:]]}

def compute_kmeans_clustering(embeddings: np.ndarray, labels: np.ndarray) -> Dict:
    kmeans = KMeans(n_clusters=18, random_state=RANDOM_STATE, n_init=10)
    predicted = kmeans.fit_predict(embeddings)
    ari = adjusted_rand_score(labels, predicted)
    print(f"Adjusted Rand Index: {ari:.4f}")
    return {'adjusted_rand_index': float(ari), 'inertia': float(kmeans.inertia_)}

print("Cluster analysis functions defined!")

In [ ]:
# @title Run Cluster Analysis
print("\n" + "="*60)
print("STEP 3: CLUSTER ANALYSIS")
print("="*60)

analysis_results = {}
comparison_summary = []

for name, result in all_embeddings.items():
    if 'embeddings_2d' in result:
        print(f"\n{'='*50}\nAnalyzing: {name}\n{'='*50}")
        embeddings = result['embeddings_2d']
        silhouette_results = compute_silhouette_scores(embeddings, labels)
        kmeans_results = compute_kmeans_clustering(embeddings, labels)
        analysis_results[name] = {'silhouette': silhouette_results, 'kmeans': kmeans_results}
        comparison_summary.append({'method': name, 'silhouette': silhouette_results['overall'],
                                   'ari': kmeans_results['adjusted_rand_index']})

comparison_summary.sort(key=lambda x: x['silhouette'], reverse=True)
print(f"\n{'='*60}\nCOMPARISON SUMMARY\n{'='*60}")
for i, item in enumerate(comparison_summary):
    print(f"{i+1}. {item['method']}: silhouette={item['silhouette']:.4f}, ARI={item['ari']:.4f}")

## Section 7: Visualization

In [ ]:
# @title Generate Visualizations
figures_dir = output_dir / 'figures'
figures_dir.mkdir(exist_ok=True)

plt.style.use('default')
n_classes = len(np.unique(labels))
color_palette = sns.color_palette("tab20", n_colors=max(n_classes, 20))

print("\n" + "="*60)
print("STEP 4: VISUALIZATION")
print("="*60)

# Plot embeddings
for name, result in list(all_embeddings.items())[:3]:  # Plot first 3 for speed
    if 'embeddings_2d' in result:
        fig, ax = plt.subplots(figsize=(10, 8))
        for label in sorted(np.unique(labels)):
            mask = labels == label
            ax.scatter(result['embeddings_2d'][mask, 0], result['embeddings_2d'][mask, 1],
                     c=[color_palette[label % 20]], label=f'Class {label}', alpha=0.5, s=15)
        ax.set_xlabel('Dimension 1'); ax.set_ylabel('Dimension 2')
        ax.set_title(f'{name.upper()}: 2D Embedding', fontweight='bold')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
        plt.tight_layout()
        fig.savefig(figures_dir / f'{name}_2d.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved {name}_2d.png")

# PCA variance plot
fig, ax = plt.subplots(figsize=(10, 6))
components = np.arange(1, len(pca_result['explained_variance_ratio']) + 1)
ax.bar(components, pca_result['explained_variance_ratio'], alpha=0.6, label='Individual')
ax.plot(components, pca_result['cumulative_variance'], 'ro-', linewidth=2, label='Cumulative')
ax.axhline(y=0.9, color='g', linestyle='--', alpha=0.7, label='90% threshold')
ax.set_xlabel('Principal Component'); ax.set_ylabel('Variance Explained')
ax.set_title('PCA: Variance Explained', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(figures_dir / 'pca_variance_explained.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved pca_variance_explained.png")

# Method comparison
fig, ax = plt.subplots(figsize=(12, 6))
methods = [item['method'] for item in comparison_summary]
x = np.arange(len(methods))
bars1 = ax.bar(x - 0.2, [item['silhouette'] for item in comparison_summary], 0.4, label='Silhouette', alpha=0.8)
bars2 = ax.bar(x + 0.2, [item['ari'] for item in comparison_summary], 0.4, label='ARI', alpha=0.8)
ax.set_xlabel('Method'); ax.set_ylabel('Score')
ax.set_title('Method Comparison', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(methods, rotation=45, ha='right')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig.savefig(figures_dir / 'method_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("Saved method_comparison_bars.png")
print(f"\nAll figures saved to {figures_dir}")

## Section 8: Summary and Download

In [ ]:
# @title Pipeline Summary
print("\n" + "="*60)
print("PHASE 2 PIPELINE SUMMARY")
print("="*60)
print(f"\nData processed:")
print(f"  Total sequences: {len(sequences)}")
print(f"  Total labels: {len(labels)}")
print(f"  Unique classes: {len(np.unique(labels))}")
print(f"\nFeature shapes:")
print(f"  K-mer features: {kmer_features.shape}")
print(f"  Positional: {positional_features.shape}")
print(f"  Dinucleotide: {dinuc_features.shape}")
print(f"\nEmbedding methods: {len(all_embeddings)}")
print(f"\nBest method: {comparison_summary[0]['method']}")
print(f"  Silhouette: {comparison_summary[0]['silhouette']:.4f}")
print(f"  ARI: {comparison_summary[0]['ari']:.4f}")
print(f"\nOutput locations:")
print(f"  Features: {output_dir}")
print(f"  Embeddings: {output_dir}")
print(f"  Figures: {figures_dir}")

In [ ]:
# @title Download Results
print("\nCreating zip file for download...")
!cd /content/drive/MyDrive && zip -r phase2_results.zip phase2_results -q
print("\n" + "="*60)
print("DOWNLOAD INSTRUCTIONS")
print("="*60)
print("\nAll results saved to Google Drive:")
print(f"  {OUTPUT_PATH}")
print("\nTo download:")
print("1. Open Google Drive in browser")
print("2. Navigate to 'phase2_results.zip'")
print("3. Download the zip file")
print("\nOr download individual files from the 'phase2_results' folder.")

## Pipeline Complete! 🎉

### What was accomplished:
1. ✅ Feature extraction (k-mer, positional, dinucleotide)
2. ✅ Dimensionality reduction (PCA, UMAP, PHATE)
3. ✅ Cluster analysis (silhouette, ARI)
4. ✅ Visualization generation

### Next Steps:
- Use best embeddings for model training (Phase 3)
- Investigate classes with low silhouette scores
- Use hierarchical clustering to guide model architecture
- Leverage confusion predictions for targeted improvements

### Key Files:
- `kmer_5_features.npy` - Feature matrix
- `labels.npy` - Label array
- Various `.npy` files - Embeddings
- `figures/*.png` - Visualizations

---

**Questions?** Check `guide.md` in the repository for detailed methodology.